# Modelling RSS Differences

Using PyMC, attempt to model the difference in RSS between sequential and non-sequential packets at a given gamma

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ble_logs import BleLogStatic

# -- Generate random date for linear regression example --

%config InlineBackend.figure_format = 'retina'
# Initialize random number generator
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")

# True parameter values
alpha, sigma = 1, 1
beta = [1, 2.5]

# Size of dataset
size = 100

# Predictor variable
X1 = np.random.randn(size)
X2 = np.random.randn(size) * 0.2

# Simulate outcome variable
Y = alpha + beta[0] * X1 + beta[1] * X2 + rng.normal(size=size) * sigma

fig, axes = plt.subplots(1, 2, sharex=True, figsize=(10, 4))
axes[0].scatter(X1, Y, alpha=0.6)
axes[1].scatter(X2, Y, alpha=0.6)
axes[0].set_ylabel("Y")
axes[0].set_xlabel("X1")
axes[1].set_xlabel("X2")

In [ ]:
import pymc as pm

print(f"Running on numpy v{np.__version__}")
print(f"Running on PyMC v{pm.__version__}")

In [ ]:
basic_model = pm.Model()

with basic_model:
    # Priors for unknown model parameters
    alpha = pm.Normal("alpha", mu=0, sigma=10)
    beta = pm.Normal("beta", mu=0, sigma=10, shape=2)
    sigma = pm.HalfNormal("sigma", sigma=1)

    # Expected value of outcome
    mu = alpha + beta[0] * X1 + beta[1] * X2

    # Likelihood (sampling distribution) of observations
    Y_obs = pm.Normal("Y_obs", mu=mu, sigma=sigma, observed=Y)

with basic_model:
    # draw 1000 posterior samples
    idata = pm.sample()

idata.posterior["alpha"].sel(draw=slice(0, 4))
az.plot_trace(idata, combined=True)

az.plot_trace(idata, combined=True)
az.summary(idata, round_to=2)

## Case Study 2

In [ ]:
# fmt: off
disaster_data = pd.Series(
    [4, 5, 4, 0, 1, 4, 3, 4, 0, 6, 3, 3, 4, 0, 2, 6,
    3, 3, 5, 4, 5, 3, 1, 4, 4, 1, 5, 5, 3, 4, 2, 5,
    2, 2, 3, 4, 2, 1, 3, np.nan, 2, 1, 1, 1, 1, 3, 0, 0,
    1, 0, 1, 1, 0, 0, 3, 1, 0, 3, 2, 2, 0, 1, 1, 1,
    0, 1, 0, 1, 0, 0, 0, 2, 1, 0, 0, 0, 1, 1, 0, 2,
    3, 3, 1, np.nan, 2, 1, 1, 1, 1, 2, 4, 2, 0, 0, 1, 4,
    0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1]
)
# fmt: on
years = np.arange(1851, 1962)

plt.plot(years, disaster_data, "o", markersize=8, alpha=0.4)
plt.ylabel("Disaster count")
plt.xlabel("Year");

In [ ]:
with pm.Model() as disaster_model:
    switchpoint = pm.DiscreteUniform("switchpoint", lower=years.min(), upper=years.max())

    # Priors for pre- and post-switch rates number of disasters
    early_rate = pm.Exponential("early_rate", 1.0)
    late_rate = pm.Exponential("late_rate", 1.0)

    # Allocate appropriate Poisson rates to years before and after current
    rate = pm.math.switch(switchpoint >= years, early_rate, late_rate)

    disasters = pm.Poisson("disasters", rate, observed=disaster_data)

    idata = pm.sample(10000)

In [ ]:
axes_arr = az.plot_trace(idata)
plt.draw()
for ax in axes_arr.flatten():
    if ax.get_title() == "switchpoint":
        labels = [label.get_text() for label in ax.get_xticklabels()]
        ax.set_xticklabels(labels, rotation=45, ha="right")
        break
plt.draw()

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(years, disaster_data, ".", alpha=0.6)
plt.ylabel("Number of accidents", fontsize=16)
plt.xlabel("Year", fontsize=16)

trace = idata.posterior.stack(draws=("chain", "draw"))

plt.vlines(trace["switchpoint"].mean(), disaster_data.min(), disaster_data.max(), color="C1")
average_disasters = np.zeros_like(disaster_data, dtype="float")
for i, year in enumerate(years):
    idx = year < trace["switchpoint"]
    average_disasters[i] = np.mean(np.where(idx, trace["early_rate"], trace["late_rate"]))

sp_hpd = az.hdi(idata, var_names=["switchpoint"])["switchpoint"].values
plt.fill_betweenx(
    y=[disaster_data.min(), disaster_data.max()],
    x1=sp_hpd[0],
    x2=sp_hpd[1],
    alpha=0.5,
    color="C1",
)
plt.plot(years, average_disasters, "k--", lw=2);

## Experimenting

In [ ]:
from arviz import plot_trace, plot_posterior

scores = [1, 2, 3, 3, 4, 4, 4, 5, 5, 5, 5, 5, 6, 6, 6, 7, 7, 8, 9]

model = pm.Model()

with model:
    # mean_prior = pm.Uniform("mean_posterior", lower=0, upper=10)
    mean_prior = pm.Normal("mean_posterior", mu=5, sigma=2)
    std_prior = pm.HalfNormal("std_posterior", sigma=2)

    likelihood = pm.Normal("obs", mu=mean_prior, sigma=std_prior, observed=scores)

    trace = pm.sample(500)

plot_trace(trace, var_names=["mean_posterior"])
plot_posterior(trace, var_names=["mean_posterior"])

plt.show()

## Bee Stuff

https://youtu.be/x2jDLDaxM74

In [ ]:
from BleFinley import *

tx_c = Transmitter('c', -1.450168, 53.368359, 122.0 + 1.5)
range_trial = BleLogStatic("/home/finley/Git/BLEanalysis/bluetooth_experiments/March 26 2025 Field Trial/Range Trials/7.log", tx_c, -1.449617, 53.370239, 111.2)
# range_trial = BleLogStatic("/home/finley/Git/BLEanalysis/bluetooth_experiments/March 26 2025 Field Trial/Range Trials/2.log", tx_c, -1.449851, 53.368798, 124.3)

In [ ]:
def rss_diff_at_gamma(ble_log :BleLog, gamma :float, packet_gap = 1) -> list[int]:
    rss_at_gamma = ble_log.all_rss_at_gamma(gamma)
    rss_differences = []

    for x in range(len(rss_at_gamma) - packet_gap):
        rss_differences.append(rss_at_gamma[x] - rss_at_gamma[x + packet_gap])

    return rss_differences

diffs = []

for degree in range(-15, 15):
    diffs.extend(rss_diff_at_gamma(range_trial, degree))

In [ ]:
# filter out all "wide" values
diffs_narrow = []

for diff in diffs:
    if 0 <= diff <= 3:
        diffs_narrow.append(diff)

bee_model = pm.Model()

with bee_model:
    # mean_prior = pm.Uniform("mean_posterior", lower=-5, upper=5)
    mean_prior = pm.Normal("mean_posterior", mu=0, sigma=3)
    std_prior = pm.HalfNormal("std_posterior", sigma=2)

    likelihood = pm.Normal("obs", mu=mean_prior, sigma=std_prior, observed=diffs_narrow)

    trace = pm.sample(10_000, chains=5)

az.plot_trace(trace, combined=True)
summary_all = az.summary(trace, round_to=2)
display(summary_all)

### Wide distribution

In [ ]:
diffs_wide = []
std_all = summary_all.loc["std_posterior", "mean"]

# filter out all "narrow" values
for diff in diffs:
    if diff <= -1 or diff >= 4:
        diffs_wide.append(diff)

# add padding values from -5 to 6
num_border_vals = diffs_wide.count(-7)
for pad in range(-6, 7):
    diffs_wide.extend([pad] * num_border_vals)

bee_wide_model = pm.Model()

with bee_wide_model:
    mean_prior = pm.Uniform("mean_posterior", lower=-5, upper=5)
    # mean_prior = pm.Normal("mean_posterior", mu=0, sigma=3)
    # std_prior = pm.HalfNormal("std_posterior", sigma=2)
    std_prior = pm.Uniform("std_posterior", lower=-17, upper=17)

    likelihood = pm.Normal("obs", mu=mean_prior, sigma=std_prior, observed=diffs_wide)

    trace_wide = pm.sample(10_000, chains=5)

az.plot_trace(trace_wide, combined=True)
summary_wide = az.summary(trace_wide, round_to=2)
display(summary_wide)

### Combined Distribution

Final model of RSS differences, combining all findings from above cells.

In [ ]:
from scipy.stats import norm

x_axis = np.linspace(-20, 20, 1000)

y_narrow = norm.pdf(x_axis, summary_all.loc["mean_posterior", "mean"], summary_all.loc["std_posterior", "mean"])
y_wide = norm.pdf(x_axis, summary_wide.loc["mean_posterior", "mean"], summary_wide.loc["std_posterior", "mean"])

# plt.plot(x_axis, y_narrow)
plt.plot(x_axis, y_wide)
plt.hist(diffs, bins=41, alpha=0.6, density=True)
plt.show()

### K-Means Clustering

Use K-Means clustering to separate the data points into two distributions.

In [ ]:
# diffs has rss diffs for +-15 degrees

freq_diff = [0] * 41 # [20] is diff of 0
rss = [-20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

for diff in diffs:
    try:
        freq_diff[diff + 20] += 1
    except:
        pass

plt.scatter(rss, freq_diff)

In [ ]:
from sklearn.cluster import KMeans

data = list(zip(rss, freq_diff))

kmeans = KMeans(n_clusters=2)
kmeans.fit(np.array(freq_diff).reshape(-1,1))

plt.scatter(rss, freq_diff, c=kmeans.labels_)

In [ ]:
# rss values in pink: -3 to 4

diffs_pink = []
std_all = summary_all.loc["std_posterior", "mean"]

# filter out all "narrow" values
for diff in diffs:
    if -3 <= diff <= 3:
        diffs_pink.append(diff)

bee_wide_pink = pm.Model()

with bee_wide_pink:
    mean_prior = pm.Uniform("mean_posterior", lower=-3, upper=3)
    # mean_prior = pm.Normal("mean_posterior", mu=0, sigma=3)
    # std_prior = pm.HalfNormal("std_posterior", sigma=2)
    std_prior = pm.Uniform("std_posterior", lower=-5, upper=5)

    likelihood = pm.Normal("obs", mu=mean_prior, sigma=std_prior, observed=diffs_pink)

    trace_pink = pm.sample(10_000, chains=5)

az.plot_trace(trace_pink, combined=True)
summary_pink = az.summary(trace_pink, round_to=2)
display(summary_pink)

In [ ]:
# rss values in yellow:  less than -3 or over 4

diffs_pink = []
std_all = summary_all.loc["std_posterior", "mean"]

# filter out all "narrow" values
for diff in diffs:
    if diff < -3 or diff > 3:
        diffs_pink.append(diff)

bee_wide_pink = pm.Model()

with bee_wide_pink:
    mean_prior = pm.Uniform("mean_posterior", lower=-10, upper=10)
    # mean_prior = pm.Normal("mean_posterior", mu=0, sigma=3)
    # std_prior = pm.HalfNormal("std_posterior", sigma=2)
    std_prior = pm.Uniform("std_posterior", lower=-17, upper=17)

    likelihood = pm.Normal("obs", mu=mean_prior, sigma=std_prior, observed=diffs_pink)

    trace_yellow = pm.sample(10_000, chains=5)

az.plot_trace(trace_yellow, combined=True)
summary_yellow = az.summary(trace_yellow, round_to=2)
display(summary_yellow)

In [ ]:
plt.hist(diffs, bins=41, alpha=0.6, density=True)

y_all = norm.pdf(x_axis, summary_pink.loc["mean_posterior", "mean"], summary_pink.loc["std_posterior", "mean"])
y_wide = norm.pdf(x_axis, summary_yellow.loc["mean_posterior", "mean"], summary_yellow.loc["std_posterior", "mean"])

plt.plot(x_axis, y_all)
plt.plot(x_axis, y_wide)

# normal over all diffs
all_mu, all_std = norm.fit(diffs)
all_norm = norm.pdf(x_axis, all_mu, all_std)
plt.plot(x_axis, all_norm)

plt.show()